# Active-Set Visualization

This notebook focuses on the active-set story behind the tiny LP. The main question is not whether the solver gets close to the right objective, but whether its returned sample is close enough to a true vertex for active-set logic and warm-start reconstruction to work reliably.


In [1]:
from __future__ import annotations

import inspect
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    for parent in ROOT.parents:
        if (parent / 'src').exists():
            ROOT = parent
            break

SRC = ROOT / 'src'
if not SRC.exists():
    raise RuntimeError(f'Could not locate the repository src directory from {Path.cwd()}')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

for module_name in tuple(sys.modules):
    if module_name == 'lpas' or module_name.startswith('lpas.'):
        sys.modules.pop(module_name, None)

try:
    import numpy as np
    from IPython.display import Markdown, display
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'Notebook kernel is missing project dependencies. '
        f"Use the project interpreter at {ROOT / '.venv' / 'bin' / 'python'}."
    ) from exc

from lpas.solver.result import ScipySolveResult

required_scipy_fields = {'nonneg_active_mask', 'augmented_primal_active_mask'}
available_scipy_fields = set(inspect.signature(ScipySolveResult).parameters)
missing_scipy_fields = required_scipy_fields - available_scipy_fields
if missing_scipy_fields:
    raise RuntimeError(
        'Notebook is importing an outdated LPAS API. '
        f'Missing ScipySolveResult fields: {sorted(missing_scipy_fields)}. '
        f"Restart the kernel with {ROOT / '.venv' / 'bin' / 'python'}."
    )

np.set_printoptions(suppress=True, precision=6)


def fmt(value, digits: int = 6) -> str:
    if value is None:
        return 'None'
    if isinstance(value, (np.bool_, bool)):
        return 'True' if bool(value) else 'False'
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.{digits}f}'
    if isinstance(value, str):
        return value
    if isinstance(value, tuple):
        return '[' + ', '.join(fmt(v, digits=digits) for v in value) + ']'
    if isinstance(value, (list, np.ndarray)):
        arr = np.asarray(value)
        if arr.ndim == 0:
            return fmt(arr.item(), digits=digits)
        return '[' + ', '.join(fmt(v, digits=digits) for v in arr.tolist()) + ']'
    return str(value)


def markdown_cell(value):
    text = fmt(value)
    return text.replace('|', '\\|').replace('\n', '<br>')


def markdown_table(headers, rows):
    lines = [
        '| ' + ' | '.join(markdown_cell(header) for header in headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(markdown_cell(cell) for cell in row) + ' |')
    return '\n'.join(lines)


def show_table(headers, rows, title: str | None = None):
    parts = []
    if title:
        parts.append(f'### {title}')
    parts.append(markdown_table(headers, rows))
    display(Markdown('\n\n'.join(parts)))


def mask_to_bits(mask) -> str:
    return ''.join('1' if bool(v) else '0' for v in mask)

from collections import Counter

from lpas.core.active_set import primal_active_mask
from lpas.experiments.generators import tiny_known_lp
from lpas.solver.adaptive_solver import AdaptiveLPSolver
from lpas.solver.scipy_handoff import solve_with_scipy
from lpas.solver.warm_start import reconstruct_from_active_set
from lpas.utils.config import SamplerConfig, SolverConfig


## Set Up the Same Solver Run

Using the same seed and configuration as the basic demo makes the interpretation consistent across notebooks.


In [2]:
problem = tiny_known_lp()
config = SolverConfig(
    batch_size=512,
    max_iter=80,
    patience=20,
    sampler=SamplerConfig(
        seed=42,
        sigma_init=2.5,
        primal_init_mean=0.75,
        dual_init_mean=0.75,
    ),
)
result = AdaptiveLPSolver(config).solve(problem)
scipy = solve_with_scipy(problem)


## Slack at the Returned Sample

The current solver configuration lands exactly on the same active vertex as SciPy, so this table checks which constraints bind and confirms that the returned point is not just near the corner but actually on it.


In [3]:
constraint_labels = ['`x1 + x2 <= 4`', '`x1 <= 2`', '`x2 <= 3`']
adaptive_slack = problem.b - problem.A @ result.best_x
scipy_slack = problem.b - problem.A @ scipy.x
slack_rows = [
    (label, fmt(a), fmt(s), fmt(a - s))
    for label, a, s in zip(constraint_labels, adaptive_slack, scipy_slack)
]
show_table(
    ['Constraint', 'Adaptive slack', 'SciPy slack', 'Adaptive - SciPy'],
    slack_rows,
    title='Distance from each boundary',
)


### Distance from each boundary

| Constraint | Adaptive slack | SciPy slack | Adaptive - SciPy |
| --- | --- | --- | --- |
| `x1 + x2 <= 4` | 0.000000 | 0.000000 | 0.000000 |
| `x1 <= 2` | 0.000000 | 0.000000 | 0.000000 |
| `x2 <= 3` | 1.000000 | 1.000000 | 0.000000 |

## Tolerance Sweep

Here I vary the activity tolerance `epsilon` and compare the primal mask for the adaptive point against the exact SciPy optimum. In the current run the mask is stable across the sweep. The bit ordering is `[x1 + x2 <= 4, x1 <= 2, x2 <= 3]`.


In [4]:
tolerances = [1e-6, 1e-4, 1e-3, 1e-2]
rows = []
for epsilon in tolerances:
    adaptive_mask = primal_active_mask(problem, result.best_x, epsilon=epsilon)
    scipy_mask = primal_active_mask(problem, scipy.x, epsilon=epsilon)
    hint = reconstruct_from_active_set(problem, adaptive_mask)
    rows.append(
        (
            f'{epsilon:.0e}',
            mask_to_bits(adaptive_mask),
            mask_to_bits(scipy_mask),
            hint.message,
            'None' if hint.objective is None else fmt(hint.objective),
        )
    )
show_table(
    ['epsilon', 'Adaptive mask', 'SciPy mask', 'Warm-start outcome', 'Recovered objective'],
    rows,
    title='Sensitivity of active-set extraction',
)


### Sensitivity of active-set extraction

| epsilon | Adaptive mask | SciPy mask | Warm-start outcome | Recovered objective |
| --- | --- | --- | --- | --- |
| 1e-06 | 110 | 110 | Feasible vertex reconstructed from active constraints | 10.000000 |
| 1e-04 | 110 | 110 | Feasible vertex reconstructed from active constraints | 10.000000 |
| 1e-03 | 110 | 110 | Feasible vertex reconstructed from active constraints | 10.000000 |
| 1e-02 | 110 | 110 | Feasible vertex reconstructed from active constraints | 10.000000 |

## Dominant Archive Patterns Over Time

The solver history stores the most common active-set pattern in the archive after each iteration. Even without a plot, the frequency table shows whether the search repeatedly visits vertex-like samples or mostly stays in the interior before the final selection.


In [5]:
pattern_counts = Counter(entry.dominant_active_pattern for entry in result.history)
pattern_rows = []
for pattern, count in pattern_counts.most_common():
    primal_pattern, dual_pattern = pattern
    pattern_rows.append(
        (
            mask_to_bits(primal_pattern),
            mask_to_bits(dual_pattern),
            count,
            f'{100.0 * count / len(result.history):.1f}%',
        )
    )
show_table(
    ['Primal bits', 'Dual bits', 'Iterations', 'Share of history'],
    pattern_rows,
    title='Dominant pattern frequencies',
)


### Dominant pattern frequencies

| Primal bits | Dual bits | Iterations | Share of history |
| --- | --- | --- | --- |
| 000 | 00 | 71 | 88.8% |
| 000 | 11 | 9 | 11.2% |

## Findings

The returned adaptive point matches the SciPy geometry exactly. The first two constraints have slack `0.000000` in both solutions, while the third keeps slack `1.000000`, so the adaptive sample is not merely close to the correct corner; it lies on the same active face pattern as the reference solver.

That conclusion is stable across the full tolerance sweep. For every tested `epsilon` from `1e-06` through `1e-02`, both methods produce primal mask `110`, and the warm-start reconstruction recovers a feasible vertex with objective `10.000000`. In this notebook, active-set extraction is therefore insensitive to the tolerance parameter.

The archive frequencies explain why this is still an informative result. The dominant history pattern is the fully inactive `000 / 00` state for `88.8%` of the run, yet the final selected sample still recovers the correct binding set. Most intermediate archive states look interior, but the best retained sample still captures the right corner geometry by the end.
